# qlora_train

Trains a LoRA adapter on OmniGen2 at 512px with the diffusion transformer loaded in NF4.
Identical to `lora_train` apart from the quantized base and the step budget: same adapter
targets, same `lora_alpha == lora_rank`, same optimizer, same data, same validation loss,
same learning rate and warmup.

Set `LORA_RANK` in section 6 and run top to bottom. One notebook run trains one rank.

Quantization covers all 309 linear modules of the diffusion transformer (nf4, double quant,
bf16 compute). The MLLM — `train.py`'s `text_encoder`, Qwen2.5-VL-3B — and the VAE stay bf16:
the MLLM is frozen and shared with the prompting arm, so quantizing it would move the
conditioning hidden states. Holding the
seven bottleneck modules in bf16 was tested in `qlora_quant_probe_q302`. At n=24 the probe
is underpowered and establishes nothing in either direction: Overall -0.281, 95% CI [-0.667, +0.106],
spanning zero but also effects well beyond the pre-registered +/-0.15 bound, so neither a difference
nor equivalence is established. SC is -0.500, nominally significant at
p 0.037 and not significant after adjusting for three metrics. Full quantization was adopted
as the simpler configuration, not because the difference was shown to be absent.

Deviations from `options/ft_lora.yml`, all listed in section 6: three resolution keys set to
512, `gradient_accumulation_steps: 8` (required by train.py's own assertion on a single GPU),
`lora_rank` per run, tensorboard-only logging, `max_train_steps: 2000` with
`checkpointing_steps: 500`, and a held-out validation loss the upstream repository does not
provide. The scheduler is `timm_constant_with_warmup` with `warmup_t: 500`, so the learning
rate is identical to the LoRA arm at every step: `max_train_steps` sets the budget, not the
schedule shape. Checkpoints 500, 1000, 1500 and 2000 sit on the same schedule as the LoRA arm at the same step numbers; only steps present in both runs can be compared directly, and the non-FSDP launch below is a further configuration difference between the arms.

Launched without `--use_fsdp`, unlike the LoRA arm. FSDP packs a wrap unit's parameters into
a single flat buffer of one dtype, and the adapter is bf16 while the bitsandbytes base
weights are uint8, so `accelerate --use_fsdp` raises `ValueError: Must flatten tensors with
uniform dtype but got torch.bfloat16 and torch.uint8` at `_validate_tensors_to_flatten`
during `accelerator.prepare`, before any training step. `--fsdp_use_orig_params True` does
not avoid this: it changes how parameters are exposed after wrapping, not whether the flat
buffer is built. On a single process `HYBRID_SHARD_ZERO2` shards across zero peer ranks, so
this is a launch-configuration difference rather than an optimization one. A probe inserted
after `accelerator.prepare` in the non-FSDP run reported `Linear4bit=309`,
`quant_state_missing=0`, `dtype=torch.uint8`.

Measured on an A100: 1.97 GiB for the quantized transformer against 7.39 GiB in bf16, and
12.79 s/step against the LoRA arm's 15.00 on the same training-only basis, so 2000 steps took 7.10 hours plus about
14 minutes of validation. Pre-clip gradient norms of 0.013 to 0.029 against
`max_grad_norm: 1`, so clipping never fires; `FIX_ACCUM_NORM` is expected to leave the trained
weights essentially unchanged, since AdamW is scale-invariant up to epsilon and the observed
gradient magnitudes sit far above it, as in the LoRA arm.

Checkpoints and logs are written to Drive, so a dropped session resumes from the last
checkpoint. Without FSDP the checkpoint is a single `model.safetensors` holding the
quantized base and the adapter together, 2.06 GB, with the 304 adapter tensors under PEFT
names such as `context_refiner.0.attn.to_k.lora_A.default.weight`. The evaluation notebook
extracts them rather than running `convert_ckpt_to_hf_format.py`, which the LoRA arm needed
for its FSDP full state dicts.


## 1. Repo and dependencies


In [ ]:
import os, subprocess, sys, json, shutil

REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/VectorSpaceLab/OmniGen2.git", REPO], check=True)
    print("cloned")
else:
    print("repo already present")

# requirements.txt omits torchdiffeq, imported by omnigen2/transport/integrators.py
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchdiffeq"], check=True)
BNB_PIN = "0.50.0"   # the version qlora_quant_probe_q302/q309 settled SKIP_MODULES=[] on
assert BNB_PIN, "set BNB_PIN before the full run"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"bitsandbytes=={BNB_PIN}"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)

# peft raises rather than returning False when torchao predates 0.16.0; Colab ships 0.10.0.
# torchao is unused by the NF4 QLoRA path.
_ao = subprocess.run([sys.executable, "-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("torchao removed")

_probe = ("from peft import LoraConfig;"
          "from peft.import_utils import is_torchao_available; is_torchao_available();"
          "from omnigen2.transport import create_transport;"
          "from omnigen2.dataset.omnigen2_train_dataset import OmniGen2TrainDataset;"
          "print('train.py imports OK')")
chk = subprocess.run([sys.executable, "-c", _probe], cwd=REPO, capture_output=True, text=True)
print(chk.stdout.strip() or chk.stderr.strip()[-1000:])
assert chk.returncode == 0, "train.py imports are broken"

sys.path.insert(0, REPO)
os.chdir(REPO)

import torch
print("gpu :", torch.cuda.get_device_name(0),
      "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
assert torch.cuda.get_device_properties(0).total_memory / 1e9 > 60, \
    "the bf16 components alone need well over 40 GB; an 80 GB A100 is required"

cloned
torchao removed
train.py imports OK
gpu : NVIDIA A100-SXM4-80GB | 85.1 GB


## 2. Drive and images

Extracted straight from Drive rather than copied locally first: holding a 48 GB archive and
48 GB of extracted files at once overflows the Colab disk when model weights are also cached.

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
IMGS, EXPECTED = "/content/imgs", 37063

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    free_gb = shutil.disk_usage("/content").free / 1e9
    print(f"free before restore: {free_gb:.0f} GB")
    assert free_gb > 70, f"only {free_gb:.0f} GB free; the image set needs ~50 GB plus model cache"
    os.makedirs(IMGS, exist_ok=True)
    # extracted straight from Drive: copying the tar locally first would hold 48 GB of archive
    # and 48 GB of extracted files at the same time, which overflows the Colab disk
    subprocess.run(f'tar -xf "{DRIVE}/imgs_subset.tar" -C {IMGS}', shell=True, check=True)
    assert n_files(IMGS) >= EXPECTED, f"restore incomplete: {n_files(IMGS)} of {EXPECTED}"
print("images:", n_files(IMGS))

for f in ["train.jsonl", "val.jsonl", "screen_exclude_from_training.txt"]:
    if not os.path.exists(f"/content/{f}"):
        subprocess.run(["cp", f"{DRIVE}/{f}", "/content/"], check=True)

print(subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout.strip().split("\n")[-1])

Mounted at /content/drive
free before restore: 193 GB
images: 37063
overlay         236G  146G   91G  62% /


## 3. Training and validation data

Converts the frozen splits into the format the training dataset expects: `task_type: "ic"`
and absolute image paths. The 62 screening targets listed in
`screen_exclude_from_training.txt` are removed from the training pool, leaving 39,591 rows.
Validation is the untouched 98-row split.

Written to Drive once and reused by every subsequent arm.


In [ ]:
LORA_DIR = f"{DRIVE}/lora"
os.makedirs(f"{LORA_DIR}/jsonls", exist_ok=True)

TRAIN_JSONL = f"{LORA_DIR}/jsonls/train_39591.jsonl"
VAL_JSONL   = f"{LORA_DIR}/jsonls/val_98.jsonl"

def convert(r):
    return {"task_type": "ic",
            "instruction": r["instruction"],
            "input_images": [os.path.join(IMGS, p) for p in r["input_images"]],
            "output_image": os.path.join(IMGS, r["output_image"])}

if not (os.path.exists(TRAIN_JSONL) and os.path.exists(VAL_JSONL)):
    excl = set(open("/content/screen_exclude_from_training.txt").read().split())
    train = [json.loads(l) for l in open("/content/train.jsonl", encoding="utf-8")]
    val   = [json.loads(l) for l in open("/content/val.jsonl", encoding="utf-8")]

    kept = [r for r in train if r["output_image"] not in excl]
    assert len(excl) == 62, f"expected 62 screen targets, got {len(excl)}"
    assert len(kept) == 39591, f"expected 39591 training rows, got {len(kept)}"
    assert len(val) == 98, f"expected 98 val rows, got {len(val)}"

    kept_t = {r["output_image"] for r in kept}
    val_t  = {r["output_image"] for r in val}
    assert not (kept_t & excl), "screen targets leaked into training"
    assert not (kept_t & val_t), "val targets leaked into training"
    print(f"train {len(train)} -> {len(kept)} after removing {len(excl)} screen targets")

    for path, rows in [(TRAIN_JSONL, kept), (VAL_JSONL, val)]:
        with open(path, "w", encoding="utf-8") as f:
            for r in rows:
                f.write(json.dumps(convert(r), ensure_ascii=False) + "\n")
        print("wrote", os.path.basename(path), len(rows), "rows")
else:
    print("jsonls already present")

# spot check that the referenced images exist
for path in [TRAIN_JSONL, VAL_JSONL]:
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    sample = rows[::max(1, len(rows) // 50)]
    missing = [p for r in sample for p in r["input_images"] + [r["output_image"]]
               if not os.path.exists(p)]
    assert not missing, f"{len(missing)} missing images in {path}, first: {missing[:2]}"
    print(f"{os.path.basename(path)}: {len(rows)} rows, sampled {len(sample)}, all images present")

import yaml

# The Drive path contains an apostrophe, which closes a single-quoted YAML scalar early.
# json.dumps emits a correctly escaped double-quoted scalar, which YAML accepts verbatim.
def _cfg(target):
    return 'data:\n  - \n    path: %s\n    type: "ic"\n    ratio: !!float 1\n' % json.dumps(target)

# rewritten only when the content differs, so concurrent runs on the same Drive folder
# never truncate a file another run is reading
def _write_if_changed(path, body):
    try:
        if open(path, encoding="utf-8").read() == body:
            return False
    except FileNotFoundError:
        pass
    open(path, "w", encoding="utf-8").write(body)
    return True

for _name, _target in [("train_ic.yml",  TRAIN_JSONL),
                       ("train_mix.yml", f"{LORA_DIR}/train_ic.yml"),
                       ("val_ic.yml",    VAL_JSONL),
                       ("val_mix.yml",   f"{LORA_DIR}/val_ic.yml")]:
    _p = f"{LORA_DIR}/{_name}"
    _changed = _write_if_changed(_p, _cfg(_target))
    _parsed = yaml.safe_load(open(_p, encoding="utf-8"))["data"][0]["path"]
    assert _parsed == _target, f"{_name} does not round-trip: {_parsed!r} != {_target!r}"
    print(f"  {_name:14s} {'written' if _changed else 'unchanged'}, parses, path round-trips")

jsonls already present
train_39591.jsonl: 39591 rows, sampled 51, all images present
val_98.jsonl: 98 rows, sampled 98, all images present
  train_ic.yml   unchanged, parses, path round-trips
  train_mix.yml  unchanged, parses, path round-trips
  val_ic.yml     unchanged, parses, path round-trips
  val_mix.yml    unchanged, parses, path round-trips


## 4. Validation loss module

`train.py` contains no validation of any kind: `val.validation_steps` is present in the
config but never read. This module computes a deterministic held-out proxy for the diffusion training objective,
with gradients disabled: it mirrors the transport, noise and timestep machinery but encodes
at the VAE posterior mode rather than a posterior sample, so successive evaluations differ
only by the model weights.

Timesteps and noise come from the repository's own `transport.sample` under a fixed seed,
so the lognormal timestep distribution and the resolution-dependent shift match training
exactly. VAE latents use the posterior mode where training samples it, which makes this a
deterministic proxy for the objective rather than the objective itself. Global RNG state is saved and restored so the training stream is unaffected.
Each row is evaluated at four independent timestep draws.


In [ ]:
VAL_MODULE = "\"\"\"Held-out validation loss for OmniGen2 training.\n\nEvaluates a deterministic held-out proxy for the training objective, using the VAE posterior mode rather than a posterior sample so successive evaluations isolate weight changes, on a held-out split with gradients disabled.\nTimesteps and noise are drawn through the repository's own transport.sample under\na fixed seed, so the timestep distribution and resolution-dependent shift match\ntraining exactly while successive evaluations differ only by the model weights.\n\nGlobal RNG state is saved and restored around every evaluation so that the\ntraining stream is unaffected.\n\nNot part of the upstream OmniGen2 repository.\n\"\"\"\n\nimport json\nimport os\nimport time\n\nimport torch\n\n\nclass ValLossEvaluator:\n    \"\"\"Per-token velocity MSE on a held-out split at fixed timesteps and noise.\"\"\"\n\n    def __init__(self, dataset, collator, transport, out_path, batch_size=2,\n                 every=250, n_repeats=4, seed=1234):\n        self.dataset = dataset\n        self.collator = collator\n        self.transport = transport\n        self.out_path = out_path\n        self.batch_size = batch_size\n        self.every = every\n        self.n_repeats = n_repeats\n        self.seed = seed\n        d = os.path.dirname(out_path)\n        if d:\n            os.makedirs(d, exist_ok=True)\n\n    @torch.no_grad()\n    def run(self, model, vae, text_encoder, freqs_cis, weight_dtype, global_step, device):\n        t_start = time.time()\n        was_training = model.training\n        model.eval()\n\n        cpu_rng = torch.get_rng_state()\n        cuda_rng = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None\n\n        sse = 0.0\n        tokens = 0.0\n        by_t = []\n\n        try:\n            for start in range(0, len(self.dataset), self.batch_size):\n                idx = list(range(start, min(start + self.batch_size, len(self.dataset))))\n                batch = self.collator([self.dataset[i] for i in idx])\n\n                text_ids = batch[\"text_ids\"].to(device)\n                text_mask = batch[\"text_mask\"].to(device)\n                text_feats = text_encoder(\n                    input_ids=text_ids,\n                    attention_mask=text_mask,\n                    output_hidden_states=False,\n                ).last_hidden_state\n\n                def encode(img):\n                    z = vae.encode(img.to(device=device, dtype=vae.dtype)).latent_dist.mode()\n                    if vae.config.shift_factor is not None:\n                        z = z - vae.config.shift_factor\n                    if vae.config.scaling_factor is not None:\n                        z = z * vae.config.scaling_factor\n                    return z.to(dtype=weight_dtype)\n\n                ref_latents = []\n                for imgs in batch[\"input_images\"]:\n                    if imgs is not None and len(imgs) > 0:\n                        ref_latents.append([encode(im).squeeze(0) for im in imgs])\n                    else:\n                        ref_latents.append(None)\n\n                x1 = [encode(im).squeeze(0) for im in batch[\"output_image\"]]\n\n                model_kwargs = dict(\n                    text_hidden_states=text_feats,\n                    text_attention_mask=text_mask,\n                    ref_image_hidden_states=ref_latents,\n                    freqs_cis=freqs_cis,\n                )\n\n                for k in range(self.n_repeats):\n                    torch.manual_seed((self.seed * 100003 + start * 101 + k) % (2 ** 31 - 1))\n                    t, x0, _ = self.transport.sample(x1, 0, 1)\n                    t, xt, ut = self.transport.path_sampler.plan(t, x0, x1)\n                    pred = model(xt, t, **dict(model_kwargs))\n                    for j in range(len(x1)):\n                        d = (pred[j].float() - ut[j].float()) ** 2\n                        sse += float(d.sum().item())\n                        tokens += float(d.numel())\n                        by_t.append((float(t[j].item()), float(d.mean().item())))\n        finally:\n            torch.set_rng_state(cpu_rng)\n            if cuda_rng is not None:\n                torch.cuda.set_rng_state_all(cuda_rng)\n            if was_training:\n                model.train()\n\n        val_loss = sse / max(tokens, 1.0)\n\n        bins = [0.0] * 4\n        counts = [0] * 4\n        for t_val, mse in by_t:\n            b = min(int(t_val * 4), 3)\n            bins[b] += mse\n            counts[b] += 1\n        binned = {f\"t_{i/4:.2f}_{(i+1)/4:.2f}\": (bins[i] / counts[i] if counts[i] else None)\n                  for i in range(4)}\n\n        with open(self.out_path, \"a\", encoding=\"utf-8\") as f:\n            f.write(json.dumps({\n                \"step\": int(global_step),\n                \"val_loss\": val_loss,\n                \"n_rows\": len(self.dataset),\n                \"n_repeats\": self.n_repeats,\n                \"n_samples\": len(by_t),\n                \"seconds\": round(time.time() - t_start, 1),\n                \"by_timestep_quartile\": binned,\n            }) + \"\\n\")\n\n        return val_loss\n"

path = f"{REPO}/omnigen2/val_loss.py"
open(path, "w", encoding="utf-8").write(VAL_MODULE)
import ast; ast.parse(open(path, encoding="utf-8").read())
print("wrote", path, f"({len(VAL_MODULE)} chars) and it parses")

wrote /content/OmniGen2/omnigen2/val_loss.py (4846 chars) and it parses


## 5. Patch train.py

Five anchored edits. Two insertions: the first builds the evaluator after the dataloader, the
second calls it every `every` steps and adds the result to the existing `logs` dict, which is
already sent to tensorboard. The call is wrapped so a validation failure cannot end a training
run. Three replacements: the gradient-norm capture, the loss normalisation at anchor `A4`
(`FIX_ACCUM_NORM`), and the anchor that matters here — `A_Q`, which swaps the shipped
`OmniGen2Transformer2DModel.from_pretrained` for an NF4 load through the `ModelMixin`
`quantization_config` route. That replacement is what makes this the QLoRA arm.

Idempotent. `git checkout -- train.py` reverts.


In [ ]:
p = f"{REPO}/train.py"
src = open(p, encoding="utf-8").read()

# num_tokens_in_batch is gathered across processes but NOT across accumulation steps. The
# "* accelerator.gradient_state.num_steps" factor in the loss normalisation is a literal
# no-op on the reference 8-GPU setup, where num_steps is 1, so it has never executed
# there; with accumulation 8 on one GPU it becomes an active multiplier and inflates the
# gradient by exactly the accumulation factor.
#
# The correction is exact rather than approximate: every target in train and val is 1024x1024
# and is resized to 512x512, so each output latent is 16x64x64 = 65,536 elements and
# local_num_tokens_in_batch (which sums output latents only) is constant at 2 x 65,536.
#   reference 8 GPU  : sum_loss / (16 x 65,536)
#   ours, unpatched  : sum_loss / ( 2 x 65,536)   = 8x
#   ours, patched    : sum_loss / (8 x 2 x 65,536) = identical to the reference
#
# True reproduces the reference gradient scale. The optimiser is AdamW, whose update
# m_hat / (sqrt(v_hat) + eps) is invariant to a constant rescaling of the loss, so the factor is
# absorbed. clip_grad_norm_(max_grad_norm=1) does not change that: pre-clip norms across every run
# span 0.0099 to 0.0768, so even unpatched an 8x gradient peaks at 0.61 and never meets the bound.
# The fix is therefore expected to have little effect on the trained weights in this gradient
# regime, through AdamW's approximate scale invariance rather than
# through clipping; it is applied so the run matches the reference gradient scale exactly.
# logs["grad_norm"] records the PRE-clip norm at every val step: the bound was hit on 0 of 60
# logged steps across the LoRA and QLoRA runs.
FIX_ACCUM_NORM = True

A1 = '    logger.info(f"{args.train.batch_size=} {args.train.gradient_accumulation_steps=} {accelerator.num_processes=} {args.train.global_batch_size=}")\n'
A2 = "                global_step += 1\n"
A3 = "                    accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)\n"
A3_NEW = "                    _gnorm = accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)\n"
A4 = "                loss = (loss * accelerator.gradient_state.num_steps * accelerator.num_processes) / num_tokens_in_batch\n"

# The transformer is the only component quantized. train.py ships no quantization path, so the
# construction is replaced with an NF4 load through the ModelMixin quantization_config route.
A_Q = ('    model = OmniGen2Transformer2DModel.from_pretrained(\n'
       '        args.model.pretrained_model_path, subfolder="transformer"\n'
       '    )\n')
A_Q_NEW = ('    # [qlora] NF4 base, adapter in bf16\n'
           '    from diffusers import BitsAndBytesConfig as _DBnB\n'
           '    _qc = _DBnB(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n'
           '                bnb_4bit_compute_dtype=torch.bfloat16,\n'
           '                bnb_4bit_use_double_quant=True)\n'
           '    model = OmniGen2Transformer2DModel.from_pretrained(\n'
           '        args.model.pretrained_model_path, subfolder="transformer",\n'
           '        quantization_config=_qc, torch_dtype=torch.bfloat16\n'
           '    )\n'
           '    import bitsandbytes as _bnb\n'
           '    _n4 = sum(1 for _m in model.modules() if isinstance(_m, _bnb.nn.Linear4bit))\n'
           '    _qs = sum(1 for _m in model.modules() if isinstance(_m, _bnb.nn.Linear4bit)\n'
           '              and getattr(_m.weight, "quant_state", None) is None)\n'
           '    print(f"[qlora] Linear4bit={_n4} quant_state_pending={_qs} bnb={_bnb.__version__}", flush=True)\n'
           '    assert _n4 == 309, f"NF4 load failed, base is not 4-bit: Linear4bit={_n4}, expected 309"\n')
A4_NEW = "                loss = (loss * accelerator.num_processes) / num_tokens_in_batch\n"

SETUP = '''    # held-out validation loss (not part of upstream OmniGen2)
    _val_eval = None
    if args.get("val_loss", None) is not None and accelerator.is_main_process:
        from omnigen2.val_loss import ValLossEvaluator
        _val_dataset = OmniGen2TrainDataset(
            args.val_loss.data_path,
            tokenizer=text_tokenizer,
            use_chat_template=args.data.use_chat_template,
            prompt_dropout_prob=0.0,
            ref_img_dropout_prob=0.0,
            max_input_pixels=OmegaConf.to_object(args.data.get("max_input_pixels", 1024 * 1024)),
            max_output_pixels=args.data.get("max_output_pixels", 1024 * 1024),
            max_side_length=args.data.get("max_side_length", 2048),
        )
        _val_eval = ValLossEvaluator(
            dataset=_val_dataset,
            collator=OmniGen2Collator(tokenizer=text_tokenizer,
                                      max_token_len=args.data.maximum_text_tokens),
            transport=transport,
            out_path=args.val_loss.out_path,
            batch_size=args.val_loss.get("batch_size", 2),
            every=args.val_loss.get("every", 250),
            n_repeats=args.val_loss.get("n_repeats", 4),
            seed=args.val_loss.get("seed", 1234),
        )
        logger.info(f"val loss: {len(_val_dataset)} rows every {_val_eval.every} steps, "
                    f"{_val_eval.n_repeats} draws per row")

'''

CALL = '''                try:
                    logs["grad_norm"] = float(_gnorm)
                except NameError:
                    pass
                if _val_eval is not None and global_step % _val_eval.every == 0:
                    try:
                        _vl = _val_eval.run(model, vae, text_encoder, freqs_cis,
                                            weight_dtype, global_step, accelerator.device)
                        logs["val_loss"] = _vl
                        logger.info("[val] step %d val_loss %.6f grad_norm %s" % (
                            global_step, _vl, logs.get("grad_norm")))
                    except Exception as _e:
                        logger.warning("[val] step %d failed: %r" % (global_step, _e))
'''

if "_val_eval" in src:
    print("already patched")
else:
    for n, a in [("1", A1), ("2", A2), ("3", A3), ("4", A4), ("Q", A_Q)]:
        assert src.count(a) == 1, f"anchor {n} found {src.count(a)} times"
    src = src.replace(A1, SETUP + A1)
    src = src.replace(A2, A2 + CALL)
    src = src.replace(A3, A3_NEW)
    src = src.replace(A_Q, A_Q_NEW)
    if FIX_ACCUM_NORM:
        src = src.replace(A4, A4_NEW)
    open(p, "w", encoding="utf-8").write(src)
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("patched and train.py parses")

now = open(p, encoding="utf-8").read()
print(f"  validation hook   : {'_val_eval.run(' in now}")
print(f"  grad-norm logging : {'_gnorm = accelerator.clip_grad_norm_' in now}")
print(f"  accum-norm fix    : {A4_NEW.strip() in now}  (FIX_ACCUM_NORM={FIX_ACCUM_NORM})")
print(f"  nf4 quantization  : {'[qlora]' in now}")
# the '_val_eval' sentinel is shared with lora_train: a train.py already patched by the LoRA arm
# in this runtime skips the block above, and the base would load in bf16 with no visible error
assert "[qlora]" in now, ("NF4 patch not applied - train.py was already patched by lora_train. "
                          "Run: git -C /content/OmniGen2 checkout -- train.py, then re-run this cell")

patched and train.py parses
  validation hook   : True
  grad-norm logging : True
  accum-norm fix    : True  (FIX_ACCUM_NORM=True)
  nf4 quantization  : True


## 6. Configuration

Every deviation from `options/ft_lora.yml` is printed below with its reason. Unlisted keys
keep their shipped values, including `learning_rate: 8e-07`, `warmup_t: 500`,
`global_batch_size: 16`, `ref_img_dropout_prob: 0.5` and `seed: 2233`. `max_train_steps` and
`checkpointing_steps` are overridden to 2000 and 500 and appear in the table below.

The reason string printed for `train.lora_rank` reads "swept 4 / 8 / 16". That sweep was run
under LoRA; only rank 8 is trained under QLoRA.

`ref_img_dropout_prob: 0.5` does not drop half the references. The dataset computes
`drop_ref_img = drop_prompt and random.random() < ref_img_dropout_prob`, so it is gated on
prompt dropout, which is 1e-4. The effective rate is 5e-5 — roughly two samples in the
32,000 this 2000-step run sees. Training is reference-conditioned on essentially every step.

In [ ]:
from omegaconf import OmegaConf

LORA_RANK = 8         # 4, 8 or 16; one run per rank
SMOKE     = False     # True: 30 steps, checkpoint at 20, val every 10. False for the real run.
VAL_EVERY = 250

RUN = f"qlora_r{LORA_RANK}" + ("_smoke" if SMOKE else "")
EXP = f"{LORA_DIR}/experiments"
os.makedirs(EXP, exist_ok=True)

# train.py writes to <repo>/experiments/<name>; symlink that to Drive so a dropped
# session resumes from the last checkpoint
link = f"{REPO}/experiments"
if os.path.islink(link) or os.path.exists(link):
    if not (os.path.islink(link) and os.readlink(link) == EXP):
        os.unlink(link) if os.path.islink(link) else shutil.rmtree(link)
        os.symlink(EXP, link)
else:
    os.symlink(EXP, link)
print("experiments ->", os.readlink(link))

base = OmegaConf.load(f"{REPO}/options/ft_lora.yml")
c    = OmegaConf.load(f"{REPO}/options/ft_lora.yml")

c.name = RUN
c.data.data_path = f"{LORA_DIR}/train_mix.yml"
c.data.max_output_pixels = 262144          # 512 * 512
c.data.max_input_pixels = [262144] * 4
c.data.max_side_length = 512
c.train.gradient_accumulation_steps = 8
c.train.lora_rank = LORA_RANK
c.logger.log_with = ["tensorboard"]
c.train.max_train_steps = 2000
c.logger.checkpointing_steps = 500

c.val_loss = {
    "data_path": f"{LORA_DIR}/val_mix.yml",
    "out_path": f"{EXP}/{RUN}/val_loss.jsonl",
    "every": 10 if SMOKE else VAL_EVERY,
    "batch_size": 2,
    "n_repeats": 4,
    "seed": 1234,
}

if SMOKE:
    c.train.max_train_steps = 30
    c.logger.checkpointing_steps = 20

CFG = f"{REPO}/options/{RUN}.yml"
OmegaConf.save(c, CFG)

reasons = {
    "data.max_output_pixels": "512x512 output, matching every generated image in the thesis",
    "data.max_input_pixels": "references at max side 512, matching load_inputs at inference",
    "data.max_side_length": "512",
    "train.gradient_accumulation_steps": "required: batch 2 x accum 8 x 1 GPU = global 16, asserted by train.py",
    "train.lora_rank": "swept 4 / 8 / 16",
    "logger.log_with": "wandb would block on an interactive login",
    "train.max_train_steps": "QLoRA arm: 2000 steps, scored at 500/1000/1500/2000",
    "logger.checkpointing_steps": "checkpoints at 500/1000/1500/2000, all four scored",
    "name": "per-run experiment directory",
    "data.data_path": "thesis training pool, screening targets removed",
}
if SMOKE:
    reasons["train.max_train_steps"] = "SMOKE: exercises data, val, checkpoint and resume"
    reasons["logger.checkpointing_steps"] = "SMOKE: forces one checkpoint write"

print(f"{'key':38s} {'shipped':<22s} -> {'this run':<22s} reason")
print("-" * 130)
for k, why in reasons.items():
    print(f"{k:38s} {str(OmegaConf.select(base, k)):<22s} -> {str(OmegaConf.select(c, k)):<22s} {why}")
print(f"{'val_loss.*':38s} {'absent':<22s} -> {'98 rows / ' + str(c.val_loss['every']) + ' steps':<22s} "
      f"upstream train.py has no validation of any kind")

print("\nunchanged:", ", ".join(
    f"{k}={OmegaConf.select(c, k)}" for k in
    ["train.learning_rate", "train.warmup_t", "train.global_batch_size", "train.batch_size",
     "train.lora_dropout", "train.gradient_checkpointing", "train.max_grad_norm",
     "data.ref_img_dropout_prob", "data.prompt_dropout_prob", "data.maximum_text_tokens", "seed"]))
print(f"\nconfig: {CFG}")
print("SMOKE RUN - roughly 8 minutes" if SMOKE else "FULL RUN - roughly 7.7 hours at the measured 13.5 s/step")

experiments -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments
key                                    shipped                -> this run               reason
----------------------------------------------------------------------------------------------------------------------------------
data.max_output_pixels                 1048576                -> 262144                 512x512 output, matching every generated image in the thesis
data.max_input_pixels                  [1048576, 1048576, 589824, 262144] -> [262144, 262144, 262144, 262144] references at max side 512, matching load_inputs at inference
data.max_side_length                   2048                   -> 512                    512
train.gradient_accumulation_steps      1                      -> 8                      required: batch 2 x accum 8 x 1 GPU = global 16, asserted by train.py
train.lora_rank                        8                      -> 8                      swept 4 / 8 / 16
log

## 7. Train

With `SMOKE = True` this is a 30-step rehearsal: it exercises data loading, the validation
hook, a checkpoint write to Drive and the gradient-norm measurement, in roughly 7 minutes.
Run section 8 before setting `SMOKE = False` for the 2000-step run, measured at 13.5 s/step
and about 7.7 hours including validation.

`resume_from_checkpoint: latest` is a shipped default, so re-running this cell after a
dropped session continues from the last checkpoint rather than restarting.


In [ ]:
import time, re

LAUNCH = ["accelerate", "launch",
          "--num_processes=1", "--num_machines=1", "--machine_rank=0", "--main_process_port=29500",
          "--mixed_precision=bf16"]

os.makedirs(f"{EXP}/{RUN}", exist_ok=True)
LOG = f"{EXP}/{RUN}/train_console.log"

t0 = time.time()
proc = subprocess.Popen(LAUNCH + ["train.py", "--config", CFG], cwd=REPO,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env={**os.environ, "WANDB_DISABLED": "true"})

with open(LOG, "a", encoding="utf-8") as lf:
    for line in proc.stdout:
        lf.write(line)
        if re.search(r"\[val\]|\[qlora\]|Saving state|steps_per_second|Traceback|Error|error|"
                     r"Number of training samples|val loss:", line):
            print(f"[{(time.time()-t0)/3600:5.2f}h] {line.rstrip()}", flush=True)
proc.wait()

print(f"\nexit {proc.returncode} after {(time.time()-t0)/3600:.2f}h")
if proc.returncode != 0:
    print("\n--- tail ---")
    print("".join(open(LOG, encoding="utf-8").readlines()[-40:]))

[ 0.00h] 2026-08-02 18:59:28.359640: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
[ 0.02h] [qlora] Linear4bit=309 quant_state_pending=0 bnb=0.50.0
[ 0.04h] 08/02/2026 19:01:23 - INFO - __main__ - [RANK 0] Number of training samples: 39591
[ 0.04h] 08/02/2026 19:01:24 - INFO - __main__ - [RANK 0] val loss: 98 rows every 250 steps, 4 draws per row
[ 0.95h] 08/02/2026 19:56:03 - INFO - __main__ - [RANK 0] [val] step 250 val_loss 0.355691 grad_norm 0.0216064453125
[ 1.86h] 08/02/2026 20:50:59 - INFO - __main__ - [RANK 0] [val] step 500 val_loss 0.355342 grad_norm 0.02001953125
[ 2.77h] 08/02/2026 21:45:17 - INFO - __main__ - [RANK 0] [val] step 750 val_loss 0.354930 grad_norm 0.0289306640625
[ 3.67h] 08/02/2026 22:39:33 - INFO - __main__ - [RANK 0] [val] step 1000 val_loss

## 8. Smoke checks and validation curve

After a smoke run this reports whether each moving part worked and what validation actually
costs. After the full run it prints the curve and applies the selection rule: the lowest
held-out loss **among steps that have a checkpoint**, since a minimum falling between
checkpoints does not make the nearest checkpoint the best available one. If the curve is
still falling at the final step the run was budget-capped rather than converged.


In [ ]:
import statistics as st, re

VL = f"{EXP}/{RUN}/val_loss.jsonl"
LOGTXT = open(LOG, encoding="utf-8").read() if os.path.exists(LOG) else ""

print("=== 1. validation hook fired ===")
if os.path.exists(VL):
    seen = {json.loads(l)["step"]: json.loads(l) for l in open(VL, encoding="utf-8")}
    rows = [seen[k] for k in sorted(seen)]
    for r in rows:
        print(f"  step {r['step']:>5d}  val_loss {r['val_loss']:.5f}  "
              f"{r['n_rows']} rows x {r['n_repeats']} draws = {r['n_samples']} samples  "
              f"{r.get('seconds', '?')}s")
    print(f"  -> {'OK' if rows else 'NO ROWS'}")
else:
    rows = []
    print("  MISSING - look for '[val] ... failed' below")
    for l in LOGTXT.split("\n"):
        if "[val]" in l:
            print("   ", l)

print("\n=== 2. validation cost (measured, not estimated) ===")
secs = [r["seconds"] for r in rows if r.get("seconds")]
if secs:
    per = st.median(secs)
    n_evals = 2000 // VAL_EVERY
    print(f"  median {per:.1f}s per evaluation")
    print(f"  full run: {n_evals} evaluations x {per:.1f}s = {n_evals*per/60:.1f} min per rank "
          f"({3*n_evals*per/3600*11.77:.0f} units across 3 ranks)")
    if per > 240:
        print("  slower than expected: consider n_repeats 2, or val batch_size 4")
else:
    print("  no timings recorded")

print("\n=== 3. gradient norm (diagnostic) ===")
gn = [float(m) for m in re.findall(r"grad_norm ([\d.]+)", LOGTXT)]
if gn:
    print(f"  pre-clip norms: {[round(x, 3) for x in gn[:8]]}")
    print(f"  median {st.median(gn):.3f} against max_grad_norm={c.train.max_grad_norm}")
    print(f"  clipping active: {st.median(gn) > c.train.max_grad_norm}")
    if not FIX_ACCUM_NORM:
        print("  NOTE: FIX_ACCUM_NORM is False, so these are 8x the true norms. The two settings")
        print("        coincide only where the logged norm exceeds 8, not 1.")
else:
    print("  no grad_norm logged - check that the section 5 patch landed")

print("\n=== 4. checkpoint written to Drive ===")
d0 = f"{EXP}/{RUN}"
ck = sorted(d for d in os.listdir(d0) if d.startswith("checkpoint-")) if os.path.isdir(d0) else []
for d in ck:
    sz = sum(os.path.getsize(os.path.join(r, f))
             for r, _, fs in os.walk(f"{d0}/{d}") for f in fs) / 1e9
    print(f"  {d}: {sz:.2f} GB")
print(f"  -> {'OK' if ck else 'NONE WRITTEN'}")

print("\n=== 5. verdict ===")
if not (os.path.exists(VL) and ck and gn):
    print("  fix the failures above before the full run")
elif SMOKE:
    print("  all checks passed - set SMOKE = False in section 6 and run the full 2000 steps")
else:
    val_at = {r["step"]: r["val_loss"] for r in rows}
    steps = sorted(int(d.split("-")[1]) for d in ck)
    have = [s for s in steps if s in val_at]
    assert have, "no validation point coincides with a checkpoint"

    print(f"\n{'step':>6s} {'val_loss':>10s} {'change':>9s}  {'ckpt':>5s}")
    prev = None
    for r in rows:
        ch = "" if prev is None else f"{r['val_loss'] - prev:+.5f}"
        mark = "yes" if r["step"] in steps else ""
        print(f"{r['step']:>6d} {r['val_loss']:>10.5f} {ch:>9s}  {mark:>5s}")
        prev = r["val_loss"]

    # selection is the argmin over checkpointed steps only: a curve minimum between
    # checkpoints does not make the nearest checkpoint the best available one
    sel = min(have, key=lambda s: val_at[s])
    curve_min = min(rows, key=lambda r: r["val_loss"])
    print(f"\ncurve minimum      : {curve_min['val_loss']:.5f} at step {curve_min['step']}")
    print(f"selected checkpoint: checkpoint-{sel}  (val_loss {val_at[sel]:.5f})")
    if curve_min["step"] not in steps:
        print(f"  note: the curve turns at {curve_min['step']}, between checkpoints; "
              f"the best available checkpoint is used")

    tail = [r["val_loss"] for r in rows[-4:]]
    print("still falling at the final step - budget-capped, not converged"
          if len(tail) >= 3 and tail[-1] == min(tail)
          else "curve has turned - the minimum is interior, so this is convergence")

    # val-loss argmin, recorded for the curve; the arm's selected checkpoint is the GPT-4.1 val argmax and is not this
    json.dump({"run": RUN, "rank": LORA_RANK,
               "curve_min_step": curve_min["step"], "curve_min_val_loss": curve_min["val_loss"],
               "selected_checkpoint": int(sel), "selected_val_loss": val_at[sel],
               "checkpoints": steps, "fix_accum_norm": FIX_ACCUM_NORM,
               "curve": [(r["step"], r["val_loss"]) for r in rows]},
              open(f"{d0}/selection.json", "w"), indent=2)
    print("wrote", f"{d0}/selection.json")

=== 1. validation hook fired ===
  step   250  val_loss 0.35569  98 rows x 4 draws = 392 samples  101.8s
  step   500  val_loss 0.35534  98 rows x 4 draws = 392 samples  101.9s
  step   750  val_loss 0.35493  98 rows x 4 draws = 392 samples  102.0s
  step  1000  val_loss 0.35482  98 rows x 4 draws = 392 samples  102.1s
  step  1250  val_loss 0.35475  98 rows x 4 draws = 392 samples  102.0s
  step  1500  val_loss 0.35471  98 rows x 4 draws = 392 samples  102.0s
  step  1750  val_loss 0.35469  98 rows x 4 draws = 392 samples  101.9s
  step  2000  val_loss 0.35466  98 rows x 4 draws = 392 samples  101.8s
  -> OK

=== 2. validation cost (measured, not estimated) ===
  median 102.0s per evaluation
  full run: 8 evaluations x 102.0s = 13.6 min per rank (8 units across 3 ranks)

=== 3. gradient norm (diagnostic) ===
  pre-clip norms: [0.022, 0.02, 0.029, 0.022, 0.025, 0.021, 0.013, 0.028]
  median 0.022 against max_grad_norm=1
  clipping active: False

=== 4. checkpoint written to Drive ===
 